In [ ]:
from run_CGAM_pipeline import run_CGAM_pipeline
from run_CGAM_pipeline import CGAMPipeline
from run_Labeling_pipeline import run_labeling_pipeline
from run_ML_inputs_pipeline import run_preprocessing_pipeline




In [ ]:
import pandas as pd

# Initialize Data Path's for CGAM and Cohen's d calculations
matched_cycles_data_path = r"..\..\SavedOutcomesAim2\Overground_EMG_Kinematics\1_FromMATLAB_Sym\matchedCycles.csv"

# Run CGAM and Cohen's d calculations
pipeline = CGAMPipeline(matched_cycles_data_path)
pipeline.load_and_filter_data(speed='FV')
pipeline.extract_feature_cols(feature_type='all')
pipeline.compute_vif_features()

cgam_df, filtered_df = pipeline.compute_cgam(['Subject','Intervention','Speed'])
cohens_df = pipeline.get_cohens_d(features=['CGAM'], timepoints=["BL","MID18","MID24","POST18","POST24","MO1FU"])

# 1000-RUN CGAM FEATURE-STABILITY 
RUN_STABILITY = True
if RUN_STABILITY:
    stability_df = pipeline.run_random_feature_stability(
        n_samples=1000,
        subset_size=15,
        groupby_cols=['Subject','Intervention','Speed']
    )


CGAM_export = False
if CGAM_export:
    orig_df = pd.read_csv(matched_cycles_data_path)
    merge_keys = ['Subject','Intervention','Speed','Trial','Cycle']
    if all(k in cgam_df.columns for k in merge_keys):
        df_appended = orig_df.merge(cgam_df[merge_keys + ['CGAM']], on=merge_keys, how='left')
        df_appended.to_csv(r"..\..\SavedOutcomesAim2\Overground_EMG_Kinematics\2_FromPython_CGAM\matchedCycles.csv", index=False)
        print("CGAM column appended and saved.")
    else:
        print("Warning: Merge keys missing, CGAM not appended.")

    cohens_df.to_csv(r"..\..\SavedOutcomesAim2\Overground_EMG_Kinematics\2_FromPython_CGAM\matchedCyclesCGAMCohens.csv", index=False)
    print("Cohen's d results exported successfully.")
else:
    print("CGAM and Cohen's d results not exported. Set CGAM_export to True to enable export.")
